# Phân tích baseline v7
Planner hướng dẫn → caller chọn tool/arg/value → summarizer chọn planner/done/give_up. Khi đã hỏi user thì summarizer bắt buộc wait. Summarizer không thay raw env_feedback bằng bản tóm tắt sinh bởi model.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import pandas as pd
from IPython.display import display, Markdown

candidates = [Path("/kaggle/working/lifemate_baseline_v7"), Path("baseline_v7/outputs"), Path("outputs"), Path("../outputs")]
ROOT = next((p for p in candidates if p.exists()), candidates[0])  # Đổi ROOT nếu report nằm chỗ khác

def jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()] if path.exists() else []

def read_run(mode):
    base = ROOT / mode
    return {
        "summary": json.loads((base/"summary.json").read_text()) if (base/"summary.json").exists() else {},
        "cases": pd.DataFrame(jsonl(base/"traces.jsonl")),
        "metrics": pd.read_csv(base/"cases.csv") if (base/"cases.csv").exists() and (base/"cases.csv").stat().st_size else pd.DataFrame(),
        "status": json.loads((base/"run_status.json").read_text()) if (base/"run_status.json").exists() else {},
    }
runs = {mode: read_run(mode) for mode in ("workflow_gold", "baseline_asr")}
for mode, run in runs.items():
    print(mode, run["status"])
    if run["summary"] and run["summary"].get("architecture") != "planner_caller_summarizer":
        print("Report thuộc kiến trúc khác:", run["summary"].get("architecture"))
    display(run["summary"])


## Metric và lỗi
`invalid_model_output`: sai contract ở plan/call/summarize. Lỗi env.validate/env.call được giữ nguyên và đưa qua summarizer; có thể kết thúc give_up hoặc quay planner. `done` là nhận định hoàn tất của summarizer, chưa phải verifier. `environment_no_progress`: read/lỗi env lặp; `control_no_progress`: planner và summarizer lặp mà không gọi tool; `duplicate_write`: code chặn write trùng đã thành công.


In [ ]:
for mode, run in runs.items():
    df = run["metrics"]
    if df.empty:
        continue
    print(mode)
    display(df[[c for c in ["id", "score_proxy", "state_hash_match", "stop_reason", "tool_count", "planner_calls", "caller_calls", "summarizer_calls", "llm_calls", "agent_latency_s"] if c in df]])
    display(df.stop_reason.value_counts().rename("count").to_frame())
    display(df.loc[df.score_proxy < 1, ["id", "error_flags", "missing_tool_names", "extra_tool_names"]])


## Chi phí ba node
Mỗi decision là một round: planner → tối đa một caller → summarizer. Planner có thể handoff trực tiếp cho summarizer. Một request N calls thành công thường cần 3N lượt LLM; các guard có thể dừng trước summarizer. Trace node_outputs có plan/call/summarize. Dùng llm_calls để so chi phí; batch giữ prompt/schema và session riêng từng mẫu.


In [ ]:
node_rows = []
for mode, run in runs.items():
    for case in run["cases"].to_dict("records"):
        for event in case.get("trace", []):
            for node, output in event.get("node_outputs", {}).items():
                node_rows.append({"mode": mode, "id": case["id"], "decision": event["decision"], "node": node,
                    **{k: output.get(k) for k in ["prompt_tokens", "generation_tokens", "latency_s"]}})
nodes = pd.DataFrame(node_rows)
if not nodes.empty:
    display(nodes.groupby(["mode", "node"])[["prompt_tokens", "generation_tokens", "latency_s"]].agg(["count", "mean", "max"]))


In [ ]:
def show_case(sample_id, mode="workflow_gold"):
    run = runs[mode]
    traces = run["cases"]
    if traces.empty or sample_id not in set(traces.id):
        print("Không tìm thấy", sample_id)
        return
    metric = run["metrics"]
    if not metric.empty:
        display(metric.loc[metric.id == sample_id].T)
    case = traces.loc[traces.id == sample_id].iloc[0].to_dict()
    print(json.dumps(case, ensure_ascii=False, indent=2))

# show_case("<sample_id>")


## Diễn giải
- Planner instruction có đúng bước còn thiếu và đúng nhánh không? Instruction không phải bằng chứng value tồn tại.
- Caller có đúng tool, field, value, namespace và bản ghi không?
- Summarizer có kiểm mọi hành động/đối tượng, kênh, điều kiện và response thật trước done không?
- Phân biệt env_feedback với current_step: cái sau chứa đề xuất model, không chứng minh thành công.
- Sau lỗi env, kiểm summarizer_output.next và bước planner sau đó có xử lý nguyên nhân lỗi hay lặp lại.
- Final_answer là kết luận cuối, không gửi về planner làm feedback; metric proxy/hash không chấm câu này.
- Khi summary nói hoàn thành nhưng thiếu write, lỗi thuộc quyết định của summarizer dù caller call hợp schema.
- Dùng wall time/throughput cho batch; workers=1 cho latency riêng. Chưa có số liệu GPU chứng minh cải thiện.
